In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 6 ###

def combine_subset_of_data_from_multiple_years_18(question_of_interest, include_2017=False):
    year_to_df = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017:
        year_to_df.append(('2017', responses_df_2017))

    records = []
    for year, df in year_to_df:
        vc = (
            df[question_of_interest]
              .value_counts(dropna=False, normalize=True)
              .mul(100)
              .round(1)
        )
        rec = vc.rename_axis(question_of_interest).reset_index(name='percentage')
        rec['year'] = year
        records.append(rec[['percentage', 'year', question_of_interest]])

    return pd.concat(records, ignore_index=True)


# --- Standardize raw DataFrames ---
question_name_alternate = 'Country'
question_name_alternate_cell18 = 'CurrentJobTitleSelect'
question_name = 'In which country do you currently reside?'

# 2022: only fix the country column, then rename a specific column if it exists
responses_df_2022_cell10[question_name] = (
    responses_df_2022_cell10[question_name]
      .replace(
          'United Kingdom (UK)',
          'United Kingdom of Great Britain and Northern Ireland',
          regex=False
      )
)
responses_df_2022_cell10.rename(
    columns={'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'},
    inplace=True
)

# 2017: map values on the original country column, then rename it
responses_df_2017[question_name_alternate] = (
    responses_df_2017[question_name_alternate]
      .replace({
          'United States': 'United States of America',
          "People 's Republic of China": 'China',
          'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
      }, regex=False)
)
responses_df_2017.rename(
    columns={
        question_name_alternate: question_name,
        question_name_alternate_cell18: (
            'Select the title most similar to your current role '
            '(or most recent title if retired): - Selected Choice'
        )
    },
    inplace=True
)

# limit to a subset of countries
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]
for df in [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]:
    df[question_name] = df[question_name].where(
        df[question_name].isin(subset_of_countries),
        'Other'
    )

# preserve this variable for backward compatibility/tests
title_for_x_axis = ''

# --- Combine and post-process ---
country_df_combined_cell18 = combine_subset_of_data_from_multiple_years_18(
    question_name,
    include_2017=True
)
# final UK label correction and sort in place
mask = (
    country_df_combined_cell18[question_name] 
    == 'United Kingdom of Great Britain and Northern Ireland'
)
country_df_combined_cell18.loc[mask, question_name] = 'United Kingdom'
country_df_combined_cell18.sort_values(by=['year', 'percentage'], inplace=True)

country_df_combined_cell18.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 77 to 0
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   percentage                                 78 non-null     float64
 1   year                                       78 non-null     object 
 2   In which country do you currently reside?  78 non-null     object 
dtypes: float64(1), object(2)
memory usage: 2.4+ KB
CPU times: user 25 ms, sys: 172 µs, total: 25.2 ms
Wall time: 25.2 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_6_try_3.pickle